# Qwen3.6-Max-Preview Cookbook

This notebook is a practical starting point for building with Qwen3.6-Max-Preview via Alibaba Cloud Model Studio's OpenAI-compatible API.

What is covered:
- setup and auth
- basic chat completions
- streaming with reasoning output
- framework integration (LangChain)
- tool-enabled agent loop
- long-context QA pattern
- structured JSON response pattern
- token usage tracking

Sources:
- https://qwen.ai/blog?id=qwen3.6-max-preview
- https://modelstudio.console.alibabacloud.com/ap-southeast-1?tab=doc#/doc/?type=model&url=2840914_2&modelId=qwen3.6-max-preview&serviceSite=international

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.
# %pip install -U openai langchain langchain-openai tiktoken

In [ ]:
import os
from openai import OpenAI

MODEL = os.getenv("DASHSCOPE_MODEL", "qwen3.6-max-preview")
BASE_URL = os.getenv("DASHSCOPE_BASE_URL", "https://dashscope-intl.aliyuncs.com/compatible-mode/v1")
API_KEY = os.getenv("DASHSCOPE_API_KEY")

if not API_KEY:
    raise ValueError("Set DASHSCOPE_API_KEY before running this notebook.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"Using model: {MODEL}")
print(f"Base URL: {BASE_URL}")

## 1) Basic Usage with Provider SDK

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise coding assistant."},
        {"role": "user", "content": "List 5 checks to harden a Python API before production launch."},
    ],
    extra_body={"enable_thinking": True},
)
print(response.choices[0].message.content)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain idempotency keys in one short example."}],
    extra_body={"enable_thinking": True},
    stream=True,
)

is_answering = False
print("\n" + "=" * 20 + " Reasoning " + "=" * 20)
for chunk in stream:
    delta = chunk.choices[0].delta
    reasoning = getattr(delta, "reasoning_content", None)
    content = getattr(delta, "content", None)

    if reasoning and not is_answering:
        print(reasoning, end="", flush=True)

    if content:
        if not is_answering:
            is_answering = True
            print("\n" + "=" * 20 + " Answer " + "=" * 20)
        print(content, end="", flush=True)
print()

### Preserve thinking across turns

The release notes describe preserve_thinking for multi-turn agent workflows.

In [ ]:
messages = [
    {"role": "system", "content": "You are a senior backend engineer."},
    {"role": "user", "content": "Draft a webhook retry strategy for payment events."},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    extra_body={"enable_thinking": True, "preserve_thinking": True},
)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "Now adapt it for high-volume traffic with strict SLAs."})

second = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    extra_body={"enable_thinking": True, "preserve_thinking": True},
)
print(second.choices[0].message.content)

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0,
)

msg = llm.invoke("Generate a rollout checklist for introducing a new coding model in CI pipelines.")
print(msg.content)

## 3) Building Agents

In [ ]:
import json

def check_service_dependency(name: str):
    status = {"postgres": "ok", "redis": "ok", "legacy-cache": "deprecated"}
    return {"dependency": name, "status": status.get(name, "unknown")}

tools = [
    {
        "type": "function",
        "function": {
            "name": "check_service_dependency",
            "description": "Return health status for a dependency.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": "You are a release assistant. Use tools before deciding."},
    {"role": "user", "content": "Can we deploy if the service depends on legacy-cache?"},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

assistant_message = first.choices[0].message
messages.append(assistant_message.model_dump())

if assistant_message.tool_calls:
    for call in assistant_message.tool_calls:
        if call.function.name == "check_service_dependency":
            args = json.loads(call.function.arguments)
            result = check_service_dependency(args["name"])
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

final = client.chat.completions.create(model=MODEL, messages=messages)
print(final.choices[0].message.content)

## 4) Advanced Applications

In [ ]:
def chunk_text(text: str, chunk_size: int = 3000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Replace with your real long document.
long_doc = """
Qwen3.6-Max-Preview improves agentic coding and instruction following.
It supports OpenAI-compatible chat completions in Model Studio.
""" * 300

chunks = chunk_text(long_doc)
context = "\n\n".join(chunks[:30])

qa = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: What rollout steps should an engineering team prioritize first?"},
    ],
)
print(qa.choices[0].message.content)

In [ ]:
# Structured output pattern without model-specific schema parameters.
# This keeps the example portable across compatible endpoints.
json_resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Return valid JSON only."},
        {"role": "user", "content": "Provide a deploy checklist with fields: risk, owner, mitigation. Return JSON array with 3 items."},
    ],
)

payload = json_resp.choices[0].message.content
print(payload)

In [ ]:
usage_demo = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Give 3 short notes on production safety for AI agents."}],
)

usage = usage_demo.usage
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)

## Model-Specific Notes

- API model id: qwen3.6-max-preview.
- OpenAI-compatible endpoint example: https://dashscope-intl.aliyuncs.com/compatible-mode/v1.
- Thinking mode is shown with extra_body {"enable_thinking": true}.
- Release notes also mention preserve_thinking for multi-turn agent tasks.
- Documented context and limits snapshot:
  - Context window: 256K
  - Maximum input: 240K (224K in thinking mode)
  - Maximum output: 64K
- Documented pricing snapshot on the model page:
  - Input: 1.3 USD per 1M tokens
  - Output: 7.8 USD per 1M tokens
- Preview caveat from release notes: model is still under active development, so behavior and quality may change between updates.
- Verify capability availability in your selected region and account before production rollout.

---
Cookbook done. Replace placeholders with your real prompts, tools, and safety checks, then run each section end-to-end in your target environment.